# Energy Consumption Forecasting Analysis
## Comprehensive Machine Learning Pipeline for Paper Publishing

This notebook implements a complete energy forecasting pipeline with:
- Time series decomposition
- Feature engineering
- Multiple machine learning models
- LSTM neural networks
- Hybrid predictions
- Comprehensive metrics (RMSE, MAE) comparison
- Advanced visualizations and analysis

## Section 1: Imports and Environment Setup

In [49]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from contextlib import redirect_stdout

warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from xgboost import XGBRegressor

print("All imports successful!")

All imports successful!


## Section 2: Global Plotting Aesthetics Setup

In [50]:
# --- Global Plotting Aesthetics Setup ---
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# Global colors for consistency
COLOR_ACTUAL = '#2c3e50'
COLOR_PREDICTED = '#e74c3c'
COLOR_LSTM = '#3498db'
COLOR_HYBRID = '#27ae60'

print("Plotting aesthetics configured!")

Plotting aesthetics configured!


In [51]:
# --- Output Folder Setup ---
output_folder = "output_plots"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created output folder: {output_folder}")
else:
    print(f"Output folder exists: {output_folder}")

def save_plot(filename, dpi=300):
    """Helper function to save plots to output folder"""
    return os.path.join(output_folder, filename)

Output folder exists: output_plots


## Section 3: TensorFlow/Keras Configuration

In [52]:
use_lstm = True
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    print("TensorFlow/Keras available - LSTM models will be trained")
except Exception as e:
    use_lstm = False
    print(f"TensorFlow/Keras unavailable: {e}")
    print("LSTM + hybrid model will be skipped.")

TensorFlow/Keras available - LSTM models will be trained


## Section 4: Evaluation Metrics Function

In [53]:
def evaluate(y_true, y_pred):
    """Calculate evaluation metrics: MAE, RMSE, and R2"""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

print("Evaluation metrics function defined!")

Evaluation metrics function defined!


## Section 5: Data Loading Functions

In [54]:
def load_household_data(path="hhblock_dataset"):
    """Load and concatenate all household CSV files from directory"""
    files = os.listdir(path)
    dfs = []
    for file in files:
        if file.endswith(".csv"):
            temp = pd.read_csv(os.path.join(path, file), low_memory=False)
            dfs.append(temp)
    df = pd.concat(dfs, ignore_index=True)
    print(f"Dataset shape: {df.shape}")
    return df

def convert_halfhour_to_timeseries(df):
    """Convert half-hourly data to time series format"""
    hh_cols = [col for col in df.columns if col.startswith("hh_")]
    df_long = df.melt(
        id_vars=["LCLid", "day"],
        value_vars=hh_cols,
        var_name="half_hour",
        value_name="energy",
    )
    df_long["day"] = pd.to_datetime(df_long["day"])
    df_long["hh_index"] = df_long["half_hour"].str.replace("hh_", "").astype(int)
    df_long["tstp"] = df_long["day"] + pd.to_timedelta(df_long["hh_index"] * 30, unit="m")
    df_long = df_long[["LCLid", "tstp", "energy"]]
    df_long.dropna(inplace=True)
    print(f"Long format shape: {df_long.shape}")
    return df_long

def batch_aggregate_energy(df_long, batch_size=100):
    """Aggregate energy consumption across households in batches"""
    unique_houses = df_long["LCLid"].unique()
    energy_batches = []
    for i in range(0, len(unique_houses), batch_size):
        batch_houses = unique_houses[i : i + batch_size]
        df_batch = df_long[df_long["LCLid"].isin(batch_houses)]
        energy_df = df_batch.groupby("tstp")["energy"].mean()
        energy_batches.append(energy_df)
        if i % 500 == 0:
            print(f"Processed houses: {i} to {min(i + batch_size, len(unique_houses))}")
    energy_df = pd.concat(energy_batches).groupby("tstp").mean()
    energy_df = energy_df.to_frame(name="energy")
    return energy_df

print("Data loading functions defined!")

Data loading functions defined!


## Section 6: Data Preprocessing Functions

In [55]:
def resample_hourly(energy_df):
    """Resample energy data to hourly frequency with interpolation"""
    hourly_df = energy_df.resample("h").mean()
    hourly_df.interpolate(method="time", inplace=True)
    print(f"Hourly dataset shape: {hourly_df.shape}")
    return hourly_df

def merge_weather(hourly_df, weather_file="weather_hourly_darksky.csv"):
    """Merge weather features with energy data"""
    weather = pd.read_csv(weather_file)
    weather["time"] = pd.to_datetime(weather["time"])
    weather.set_index("time", inplace=True)
    weather = weather[["temperature", "humidity", "windSpeed", "pressure"]]
    hourly_df = hourly_df.merge(weather, left_index=True, right_index=True, how="left")
    hourly_df.interpolate(method="time", inplace=True)
    print(f"Dataset after weather merge: {hourly_df.shape}")
    return hourly_df

def add_holidays(hourly_df, holidays_file="uk_bank_holidays.csv"):
    """Add binary feature for UK bank holidays"""
    holidays = pd.read_csv(holidays_file)
    holidays["Bank holidays"] = pd.to_datetime(holidays["Bank holidays"])
    holiday_dates = holidays["Bank holidays"].dt.date
    hourly_df["is_holiday"] = pd.Series(hourly_df.index.date).isin(holiday_dates).astype(int).values
    print(f"Dataset after holiday feature: {hourly_df.shape}")
    return hourly_df

print("Data preprocessing functions defined!")

Data preprocessing functions defined!


## Section 7: Feature Engineering

In [56]:
def feature_engineering(hourly_df):
    """Create engineered features for machine learning models"""
    # Temporal features
    hourly_df["hour"] = hourly_df.index.hour
    hourly_df["day"] = hourly_df.index.day
    hourly_df["month"] = hourly_df.index.month
    hourly_df["dayofweek"] = hourly_df.index.dayofweek
    hourly_df["is_weekend"] = (hourly_df["dayofweek"] >= 5).astype(int)
    
    # Cyclical encoding for hour and month
    hourly_df["hour_sin"] = np.sin(2 * np.pi * hourly_df["hour"] / 24)
    hourly_df["hour_cos"] = np.cos(2 * np.pi * hourly_df["hour"] / 24)
    hourly_df["month_sin"] = np.sin(2 * np.pi * hourly_df["month"] / 12)
    hourly_df["month_cos"] = np.cos(2 * np.pi * hourly_df["month"] / 12)
    
    # Lag features
    hourly_df["lag1"] = hourly_df["energy"].shift(1)
    hourly_df["lag24"] = hourly_df["energy"].shift(24)
    hourly_df["lag168"] = hourly_df["energy"].shift(168)  # Weekly lag
    
    # Rolling statistics
    hourly_df["roll_mean3"] = hourly_df["energy"].rolling(3).mean()
    hourly_df["roll_mean6"] = hourly_df["energy"].rolling(6).mean()
    hourly_df["roll_std6"] = hourly_df["energy"].rolling(6).std()
    
    hourly_df.dropna(inplace=True)
    print(f"Dataset after feature engineering: {hourly_df.shape}")
    print(f"Total features created: {hourly_df.shape[1] - 1}")  # -1 for index
    return hourly_df

print("Feature engineering function defined!")

Feature engineering function defined!


## Section 8: Load and Process Data

In [57]:
# --- DATA LOADING AND PREPROCESSING ---
print("Starting data loading and preprocessing...\n")

# Load household data
df = load_household_data("hhblock_dataset")
print(f"\nFirst few rows of raw data:")
print(df.head())

# Convert half-hourly to time series
df_long = convert_halfhour_to_timeseries(df)
print(f"\nTime series format sample:")
print(df_long.head())

Starting data loading and preprocessing...

Dataset shape: (3469352, 50)

First few rows of raw data:
       LCLid         day   hh_0   hh_1   hh_2   hh_3   hh_4   hh_5   hh_6  \
0  MAC000002  2012-10-13  0.263  0.269  0.275  0.256  0.211  0.136  0.161   
1  MAC000002  2012-10-14  0.262  0.166  0.226  0.088  0.126  0.082  0.123   
2  MAC000002  2012-10-15  0.192  0.097  0.141  0.083  0.132  0.070  0.130   
3  MAC000002  2012-10-16  0.237  0.237  0.193  0.118  0.098  0.107  0.094   
4  MAC000002  2012-10-17  0.157  0.211  0.155  0.169  0.101  0.117  0.084   

    hh_7  ...  hh_38  hh_39  hh_40  hh_41  hh_42  hh_43  hh_44  hh_45  hh_46  \
0  0.119  ...  0.918  0.278  0.267  0.239  0.230  0.233  0.235  0.188  0.259   
1  0.083  ...  1.075  0.956  0.821  0.745  0.712  0.511  0.231  0.210  0.278   
2  0.074  ...  1.164  0.249  0.225  0.258  0.260  0.334  0.299  0.236  0.241   
3  0.109  ...  0.966  0.172  0.192  0.228  0.203  0.211  0.188  0.213  0.157   
4  0.118  ...  0.223  0.075  0.230 

In [58]:
# Aggregate energy consumption
energy_df = batch_aggregate_energy(df_long, batch_size=100)
print(f"\nAggregated energy data:")
print(energy_df.head())
print(f"Energy statistics:")
print(energy_df.describe())

Processed houses: 0 to 100
Processed houses: 500 to 600
Processed houses: 1000 to 1100
Processed houses: 1500 to 1600
Processed houses: 2000 to 2100
Processed houses: 2500 to 2600
Processed houses: 3000 to 3100
Processed houses: 3500 to 3600
Processed houses: 4000 to 4100
Processed houses: 4500 to 4600
Processed houses: 5000 to 5100
Processed houses: 5500 to 5560

Aggregated energy data:
                       energy
tstp                         
2011-11-24 00:00:00  0.206815
2011-11-24 00:30:00  0.284111
2011-11-24 01:00:00  0.240204
2011-11-24 01:30:00  0.253296
2011-11-24 02:00:00  0.188741
Energy statistics:
             energy
count  39696.000000
mean       0.216462
std        0.076430
min        0.086279
25%        0.161188
50%        0.208580
75%        0.259694
max        0.501911


In [59]:
# Resample to hourly frequency
hourly_df = resample_hourly(energy_df)
print(f"\nHourly data:")
print(hourly_df.head())

# Merge weather features
hourly_df = merge_weather(hourly_df, weather_file="weather_hourly_darksky.csv")
print(f"\nAfter weather merge:")
print(hourly_df.head())
print(f"\nMissing values:")
print(hourly_df.isnull().sum())

Hourly dataset shape: (19848, 1)

Hourly data:
                       energy
tstp                         
2011-11-24 00:00:00  0.245463
2011-11-24 01:00:00  0.246750
2011-11-24 02:00:00  0.170926
2011-11-24 03:00:00  0.147333
2011-11-24 04:00:00  0.129981
Dataset after weather merge: (19848, 5)

After weather merge:
                       energy  temperature  humidity  windSpeed  pressure
tstp                                                                     
2011-11-24 00:00:00  0.245463         9.01      0.96       2.92   1029.52
2011-11-24 01:00:00  0.246750         8.99      0.94       2.67   1029.40
2011-11-24 02:00:00  0.170926         8.56      0.96       2.65   1029.38
2011-11-24 03:00:00  0.147333         9.03      0.96       3.05   1029.18
2011-11-24 04:00:00  0.129981         8.78      0.96       3.06   1028.89

Missing values:
energy         0
temperature    0
humidity       0
windSpeed      0
pressure       0
dtype: int64


In [60]:
# Add holiday feature
hourly_df = add_holidays(hourly_df, holidays_file="uk_bank_holidays.csv")

# Feature engineering
hourly_df = feature_engineering(hourly_df)
print(f"\nFinal dataset with all features:")
print(hourly_df.head())
print(f"\nDataset info:")
print(hourly_df.info())

Dataset after holiday feature: (19848, 6)
Dataset after feature engineering: (19680, 21)
Total features created: 20

Final dataset with all features:
                       energy  temperature  humidity  windSpeed  pressure  \
tstp                                                                        
2011-12-01 00:00:00  0.189959         9.44      0.91       6.97   1009.98   
2011-12-01 01:00:00  0.175000        10.38      0.92       7.65   1008.27   
2011-12-01 02:00:00  0.163441        11.43      0.90       8.04   1006.93   
2011-12-01 03:00:00  0.119772        11.54      0.85       7.50   1006.77   
2011-12-01 04:00:00  0.121612        11.48      0.80       6.43   1006.76   

                     is_holiday  hour  day  month  dayofweek  ...  hour_sin  \
tstp                                                          ...             
2011-12-01 00:00:00           0     0    1     12          3  ...  0.000000   
2011-12-01 01:00:00           0     1    1     12          3  ...  0.2588

## Section 9: Train Traditional Machine Learning Models

In [61]:
def train_and_evaluate_models(X, y):
    """Train multiple ML models with time series cross-validation"""
    models = {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42, n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
        "XGBoost": XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=42),
    }

    results = {}
    tscv = TimeSeriesSplit(n_splits=5)
    
    all_predictions = {}  # Store predictions for visualization

    for name, model in models.items():
        print(f"\n{'='*50}")
        print(f"Training {name}...")
        print(f"{'='*50}")
        preds = []
        actual = []
        split_num = 0
        
        for train_index, test_index in tscv.split(X):
            split_num += 1
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            preds.extend(pred)
            actual.extend(y_test)
            
            fold_metrics = evaluate(y_test, pred)
            print(f"Fold {split_num}: MAE={fold_metrics['MAE']:.4f}, RMSE={fold_metrics['RMSE']:.4f}")
        
        results[name] = evaluate(actual, preds)
        all_predictions[name] = (actual, preds)
        print(f"\nOverall {name} Results:")
        print(f"  MAE:  {results[name]['MAE']:.4f}")
        print(f"  RMSE: {results[name]['RMSE']:.4f}")
        print(f"  R2:   {results[name]['R2']:.4f}")

    results_df = pd.DataFrame(results).T
    print(f"\n{'='*50}")
    print("SUMMARY OF ALL MODELS")
    print(f"{'='*50}")
    print(results_df)

    # Pick best model by least RMSE
    best_model_name = results_df["RMSE"].idxmin()
    best_model = models[best_model_name]
    best_model.fit(X, y)  # final full fit for inference
    
    print(f"\n*** Best Model Selected: {best_model_name} (RMSE: {results_df.loc[best_model_name, 'RMSE']:.4f}) ***")

    return best_model_name, best_model, results_df, all_predictions

print("Model training function defined!")

Model training function defined!


In [62]:
# --- PREPARE DATA FOR MODELING ---
feature_list = [
    "hour", "day", "month", "dayofweek", "is_weekend", "is_holiday",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "lag1", "lag24", "lag168", "roll_mean3", "roll_mean6", "roll_std6",
    "temperature", "humidity", "windSpeed", "pressure",
]

X = hourly_df[feature_list]
y = hourly_df["energy"]

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nFeatures used:")
for i, feat in enumerate(feature_list, 1):
    print(f"  {i:2d}. {feat}")


Feature matrix shape: (19680, 20)
Target variable shape: (19680,)

Features used:
   1. hour
   2. day
   3. month
   4. dayofweek
   5. is_weekend
   6. is_holiday
   7. hour_sin
   8. hour_cos
   9. month_sin
  10. month_cos
  11. lag1
  12. lag24
  13. lag168
  14. roll_mean3
  15. roll_mean6
  16. roll_std6
  17. temperature
  18. humidity
  19. windSpeed
  20. pressure


In [63]:
# Train and evaluate all models
best_model_name, best_model, results_df, all_predictions = train_and_evaluate_models(X, y)


Training LinearRegression...
Fold 1: MAE=0.0068, RMSE=0.0088
Fold 2: MAE=0.0081, RMSE=0.0105
Fold 3: MAE=0.0081, RMSE=0.0105
Fold 4: MAE=0.0058, RMSE=0.0075
Fold 5: MAE=0.0080, RMSE=0.0104

Overall LinearRegression Results:
  MAE:  0.0073
  RMSE: 0.0096
  R2:   0.9829

Training RandomForest...
Fold 1: MAE=0.0075, RMSE=0.0096
Fold 2: MAE=0.0064, RMSE=0.0087
Fold 3: MAE=0.0056, RMSE=0.0076
Fold 4: MAE=0.0036, RMSE=0.0049
Fold 5: MAE=0.0043, RMSE=0.0060

Overall RandomForest Results:
  MAE:  0.0055
  RMSE: 0.0076
  R2:   0.9895

Training GradientBoosting...
Fold 1: MAE=0.0071, RMSE=0.0093
Fold 2: MAE=0.0074, RMSE=0.0098
Fold 3: MAE=0.0068, RMSE=0.0090
Fold 4: MAE=0.0047, RMSE=0.0063
Fold 5: MAE=0.0063, RMSE=0.0085

Overall GradientBoosting Results:
  MAE:  0.0065
  RMSE: 0.0087
  R2:   0.9862

Training XGBoost...
Fold 1: MAE=0.0080, RMSE=0.0101
Fold 2: MAE=0.0057, RMSE=0.0077
Fold 3: MAE=0.0047, RMSE=0.0064
Fold 4: MAE=0.0029, RMSE=0.0038
Fold 5: MAE=0.0037, RMSE=0.0050

Overall XGBoost 

## Section 10: Feature Importance Analysis

In [64]:
def compute_feature_importance(model, features):
    """Extract and visualize feature importance from tree-based models"""
    importance = model.feature_importances_
    feat_df = pd.DataFrame({"Feature": features, "Importance": importance}).sort_values(by="Importance", ascending=False)
    
    plt.figure(figsize=(12, 8))
    ax = sns.barplot(x="Importance", y="Feature", data=feat_df, palette="viridis")
    plt.title("Feature Importance Ranking", weight='bold', pad=20, fontsize=18)
    plt.xlabel("Importance Score", weight='bold', fontsize=14)
    plt.ylabel("Features", weight='bold', fontsize=14)
    sns.despine(left=True, bottom=True)
    
    # Add value annotations
    for p in ax.patches:
        ax.annotate(f'{p.get_width():.4f}', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', xytext=(5, 0), textcoords='offset points', fontsize=9)
        
    plt.tight_layout()
    plt.savefig(save_plot("feature_importance.png"), dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved feature importance plot")
    return feat_df

# Compute feature importance
if hasattr(best_model, 'feature_importances_'):
    feat_df = compute_feature_importance(best_model, feature_list)
    print("\nTop 10 Most Important Features:")
    print(feat_df.head(10))
else:
    print(f"Note: {best_model_name} does not support feature importance extraction")

Saved feature importance plot

Top 10 Most Important Features:
       Feature  Importance
11       lag24    0.608205
12      lag168    0.193060
13  roll_mean3    0.082081
10        lag1    0.035531
0         hour    0.021680
7     hour_cos    0.021010
6     hour_sin    0.014669
9    month_cos    0.004001
15   roll_std6    0.003730
4   is_weekend    0.003285


## Section 11: SHAP Explainability Analysis

In [65]:
def shap_explain(model, X):
    """Generate SHAP summary plots for model explainability"""
    try:
        import shap
        print("Generating SHAP explanations...")
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X.iloc[:1000])  # Use subset for speed
        
        plt.figure()
        shap.summary_plot(shap_values, X.iloc[:1000], show=False)
        plt.savefig(save_plot("shap_summary.png"), dpi=300, bbox_inches='tight')
        plt.close()
        print("Saved SHAP summary plot")
    except Exception as e:
        print(f"SHAP explainability skipped: {e}")

# Generate SHAP explanations
if hasattr(best_model, 'feature_importances_'):
    shap_explain(best_model, X)
else:
    print("Skipping SHAP analysis - only works with tree-based models")

Generating SHAP explanations...
Saved SHAP summary plot


## Section 12: RMSE and MAE Comparison Plots

In [83]:
def plot_rmse_mae_comparison(results_df, filename_rmse="model_comparison_rmse.png", filename_mae="model_comparison_mae.png"):
    """Create comparison plots for RMSE and MAE across models"""
    results_df_reset = results_df.reset_index().rename(columns={"index": "Model"})
    
    # Define elegant color palette
    palette = sns.color_palette("Set2", len(results_df_reset))
    
    # RMSE Comparison
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.barplot(data=results_df_reset, x="RMSE", y="Model", palette=palette, ax=ax)
    
    ax.set_xlabel("RMSE (kWh)", weight='bold', fontsize=14)
    ax.set_ylabel("Model", weight='bold', fontsize=14)
    ax.set_title("Root Mean Squared Error (RMSE) Comparison", weight='bold', fontsize=16, pad=20)
    sns.despine(left=False, bottom=True)
    
    # Add value annotations
    for i, bar in enumerate(ax.patches):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2., f' {width:.4f}',
                ha='left', va='center', fontsize=11, weight='bold')
    
    plt.tight_layout()
    plt.savefig(save_plot(filename_rmse), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved RMSE comparison plot")
    
    # MAE Comparison
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.barplot(data=results_df_reset, x="MAE", y="Model", palette=palette, ax=ax)
    
    ax.set_xlabel("MAE (kWh)", weight='bold', fontsize=14)
    ax.set_ylabel("Model", weight='bold', fontsize=14)
    ax.set_title("Mean Absolute Error (MAE) Comparison", weight='bold', fontsize=16, pad=20)
    sns.despine(left=False, bottom=True)
    
    # Add value annotations
    for i, bar in enumerate(ax.patches):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2., f' {width:.4f}',
                ha='left', va='center', fontsize=11, weight='bold')
    
    plt.tight_layout()
    plt.savefig(save_plot(filename_mae), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved MAE comparison plot")


# Generate RMSE and MAE comparison plots
plot_rmse_mae_comparison(results_df)

Saved RMSE comparison plot
Saved MAE comparison plot


In [71]:
# Combined RMSE and MAE in one figure for paper
def plot_combined_metrics_comparison(results_df, filename="model_comparison_combined.png"):
    """Create a combined comparison plot showing both RMSE and MAE"""
    results_df_reset = results_df.reset_index().rename(columns={"index": "Model"})
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    palette = sns.color_palette("Set2", len(results_df_reset))
    
    # RMSE
    sns.barplot(data=results_df_reset, x="Model", y="RMSE", palette=palette, ax=ax1)
    ax1.set_ylabel("RMSE (kWh)", weight='bold', fontsize=12)
    ax1.set_xlabel("Model", weight='bold', fontsize=12)
    ax1.set_title("RMSE Comparison", weight='bold', fontsize=14, pad=15)
    ax1.tick_params(axis='x', rotation=45)
    sns.despine(ax=ax1)
    
    for bar in ax1.patches:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=10, weight='bold')
    
    # MAE
    sns.barplot(data=results_df_reset, x="Model", y="MAE", palette=palette, ax=ax2)
    ax2.set_ylabel("MAE (kWh)", weight='bold', fontsize=12)
    ax2.set_xlabel("Model", weight='bold', fontsize=12)
    ax2.set_title("MAE Comparison", weight='bold', fontsize=14, pad=15)
    ax2.tick_params(axis='x', rotation=45)
    sns.despine(ax=ax2)
    
    for bar in ax2.patches:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=10, weight='bold')
    
    plt.suptitle("Model Performance Metrics Comparison", fontsize=16, weight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(save_plot(filename), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved combined metrics plot to {filename}")

plot_combined_metrics_comparison(results_df)

Saved combined metrics plot to model_comparison_combined.png


## Section 13: LSTM Data Preparation and Training

In [66]:
def prepare_lstm_data(hourly_df, target="energy", seq_len=24):
    """Prepare sequential data for LSTM training"""
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(hourly_df[[target]])

    X_seq, y_seq = [], []
    for i in range(len(scaled) - seq_len):
        X_seq.append(scaled[i : i + seq_len])
        y_seq.append(scaled[i + seq_len])
    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)

    split = int(len(X_seq) * 0.8)
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]
    
    print(f"LSTM Training data shape: {X_train.shape}")
    print(f"LSTM Testing data shape: {X_test.shape}")
    
    return X_train, X_test, y_train, y_test, scaler

def train_lstm(X_train, y_train, seq_len=24, epochs=25):
    """Build and train LSTM model"""
    model_lstm = Sequential([
        LSTM(128, return_sequences=True, input_shape=(seq_len, 1)),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dense(1),
    ])
    model_lstm.compile(optimizer="adam", loss="mse")
    print("\nLSTM Model Architecture:")
    model_lstm.summary()
    
    print(f"\nTraining LSTM for {epochs} epochs...")
    history = model_lstm.fit(X_train, y_train, epochs=epochs, batch_size=64, 
                             validation_split=0.1, verbose=0)
    print("✓ LSTM training completed")
    
    return model_lstm, history

print("LSTM preparation and training functions defined!")

LSTM preparation and training functions defined!


In [67]:
# Prepare data for LSTM
if use_lstm:
    print("Preparing LSTM data...\n")
    X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm, scaler_lstm = prepare_lstm_data(hourly_df, target="energy", seq_len=24)
else:
    print("Skipping LSTM data preparation - TensorFlow not available")

Preparing LSTM data...

LSTM Training data shape: (15724, 24, 1)
LSTM Testing data shape: (3932, 24, 1)


In [68]:
# Train LSTM model
if use_lstm:
    model_lstm, history_lstm = train_lstm(X_train_lstm, y_train_lstm, seq_len=24, epochs=25)
else:
    print("Skipping LSTM training")


LSTM Model Architecture:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_6 (LSTM)                   │ (None, 24, 128)        │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 24, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 128,417 (501.63 KB)

 Trainable params: 128,417 (501.63 KB)

 Non-trainable params: 0 (0.00 B)


Training LSTM for 25 epochs...
✓ LSTM training completed


## Section 14: LSTM and Hybrid Model Predictions & Evaluation

In [69]:
lstm_results = None
hybrid_results = None
y_test_inv = None
lstm_pred_inv = None
hybrid_pred = None

if use_lstm:
    print("Generating predictions...\n")
    
    # LSTM predictions
    lstm_pred = model_lstm.predict(X_test_lstm, verbose=0)
    lstm_pred_inv = scaler_lstm.inverse_transform(lstm_pred)
    y_test_inv = scaler_lstm.inverse_transform(y_test_lstm)
    lstm_results = evaluate(y_test_inv.flatten(), lstm_pred_inv.flatten())
    
    print(f"{'='*50}")
    print("LSTM MODEL RESULTS")
    print(f"{'='*50}")
    print(f"MAE:  {lstm_results['MAE']:.4f}")
    print(f"RMSE: {lstm_results['RMSE']:.4f}")
    print(f"R2:   {lstm_results['R2']:.4f}")
    
    # Hybrid model (average of best traditional model and LSTM)
    best_pred = best_model.predict(X.iloc[-len(y_test_inv) :])
    hybrid_pred = (best_pred + lstm_pred_inv.flatten()) / 2
    hybrid_results = evaluate(y_test_inv.flatten(), hybrid_pred)
    
    print(f"\n{'='*50}")
    print(f"HYBRID MODEL RESULTS ({best_model_name} + LSTM)")
    print(f"{'='*50}")
    print(f"MAE:  {hybrid_results['MAE']:.4f}")
    print(f"RMSE: {hybrid_results['RMSE']:.4f}")
    print(f"R2:   {hybrid_results['R2']:.4f}")
else:
    print("Skipping LSTM and hybrid predictions")

Generating predictions...

LSTM MODEL RESULTS
MAE:  0.0076
RMSE: 0.0098
R2:   0.9836

HYBRID MODEL RESULTS (XGBoost + LSTM)
MAE:  0.0042
RMSE: 0.0054
R2:   0.9950


## Section 15: Results Visualization - Actual vs Predicted

In [72]:
def plot_results(y_actual, y_pred, filename="actual_vs_predicted.png", title="Actual vs Predicted Energy"):
    """Plot actual vs predicted values"""
    fig, ax = plt.subplots(figsize=(16, 6))
    y_actual_flat = np.array(y_actual).flatten()
    y_pred_flat = np.array(y_pred).flatten()

    # Plot first 300 points for clarity
    time_points = range(min(300, len(y_actual_flat)))
    ax.plot(time_points, y_actual_flat[:300], label="Actual Load", color=COLOR_ACTUAL, linewidth=2.5, alpha=0.9, marker='o', markersize=3)
    ax.plot(time_points, y_pred_flat[:300], label="Predicted Load", color=COLOR_PREDICTED, linewidth=2.5, alpha=0.9, linestyle="--", marker='s', markersize=3)
    
    # Fill between for visual effect
    ax.fill_between(time_points, y_actual_flat[:300], alpha=0.1, color=COLOR_ACTUAL)
    
    ax.legend(loc="upper right", frameon=True, shadow=True, fontsize=12)
    ax.set_xlabel("Time Index (Hours)", weight='bold', fontsize=12)
    ax.set_ylabel("Energy Consumption (kWh)", weight='bold', fontsize=12)
    ax.set_title(title, weight='bold', fontsize=14, pad=20)
    sns.despine()
    plt.tight_layout()
    plt.savefig(save_plot(filename), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved plot to {filename}")

# Plot hybrid predictions
if use_lstm and hybrid_pred is not None:
    plot_results(y_test_inv, hybrid_pred, filename="hybrid_prediction.png", 
                title=f"Actual vs Hybrid Prediction ({best_model_name} + LSTM)")
    plot_results(y_test_inv, lstm_pred_inv, filename="lstm_prediction.png", 
                title="Actual vs LSTM Prediction")

Saved plot to hybrid_prediction.png
Saved plot to lstm_prediction.png


## Section 16: Time Series Decomposition

In [73]:
# Seasonal decomposition
try:
    from statsmodels.tsa.seasonal import seasonal_decompose
    print("Performing seasonal decomposition...")
    decomp = seasonal_decompose(hourly_df["energy"], period=24)
    fig = decomp.plot()
    fig.set_size_inches(14, 10)
    for ax in fig.axes:
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.set_ylabel(ax.get_ylabel(), weight='bold')
    plt.suptitle("Seasonal Decomposition of Energy Time Series\n(Trend, Seasonal, Residual Components)", 
                 weight='bold', fontsize=14, y=0.995)
    plt.tight_layout()
    plt.savefig(save_plot("seasonal_decompose.png"), dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved seasonal decomposition")
except Exception as e:
    print(f"Seasonal decomposition skipped: {e}")

Performing seasonal decomposition...
Saved seasonal decomposition


## Section 17: Autocorrelation Function (ACF) Analysis

In [74]:
# ACF plot
try:
    from statsmodels.graphics.tsaplots import plot_acf
    print("Plotting autocorrelation function...")
    fig, ax = plt.subplots(figsize=(14, 5))
    plot_acf(hourly_df["energy"], lags=48, ax=ax, color='#1f77b4', markersize=6, title="")
    ax.set_title("Autocorrelation Function (ACF) - Lags up to 48 Hours", weight='bold', fontsize=14, pad=15)
    ax.set_xlabel("Lags (Hours)", weight='bold', fontsize=12)
    ax.set_ylabel("Autocorrelation", weight='bold', fontsize=12)
    sns.despine()
    plt.tight_layout()
    plt.savefig(save_plot("acf_plot.png"), dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved ACF plot")
except Exception as e:
    print(f"ACF plot skipped: {e}")

Plotting autocorrelation function...
Saved ACF plot


## Section 18: Load Duration Curve

In [75]:
# Load Duration Curve
print("Creating load duration curve...")
sorted_load = np.sort(hourly_df["energy"])[::-1]
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sorted_load, color='#8e44ad', linewidth=2.5, label='Load Duration')
ax.fill_between(range(len(sorted_load)), sorted_load, color='#8e44ad', alpha=0.2)
ax.set_title("Load Duration Curve", weight='bold', fontsize=14, pad=15)
ax.set_xlabel("Time (Sorted Hours)", weight='bold', fontsize=12)
ax.set_ylabel("Energy Consumption (kWh)", weight='bold', fontsize=12)
ax.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(save_plot("load_duration_curve.png"), dpi=300, bbox_inches='tight')
plt.show()
print("Saved load duration curve")

Creating load duration curve...
Saved load duration curve


## Section 19: Temperature vs Energy Correlation

In [76]:
# Temperature vs Energy scatter plot
print("Creating temperature vs energy plot...")
fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(hourly_df["temperature"], hourly_df["energy"], 
                      c=hourly_df["temperature"], cmap='coolwarm', alpha=0.6, 
                      edgecolors='white', linewidth=0.5, s=30)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Temperature (°C)', weight='bold')

# Add trend line
z = np.polyfit(hourly_df["temperature"], hourly_df["energy"], 2)
p = np.poly1d(z)
x_trend = np.linspace(hourly_df["temperature"].min(), hourly_df["temperature"].max(), 100)
ax.plot(x_trend, p(x_trend), "r--", linewidth=2.5, alpha=0.8, label='Polynomial Trend')

ax.set_title("Energy Consumption vs Temperature", weight='bold', fontsize=14, pad=15)
ax.set_xlabel("Temperature (°C)", weight='bold', fontsize=12)
ax.set_ylabel("Energy Consumption (kWh)", weight='bold', fontsize=12)
ax.legend(frameon=True, shadow=True)
sns.despine()
plt.tight_layout()
plt.savefig(save_plot("temp_vs_energy.png"), dpi=300, bbox_inches='tight')
plt.show()
print("Saved temperature vs energy plot")

Creating temperature vs energy plot...
Saved temperature vs energy plot


## Section 20: Residual Error Distribution Analysis

In [77]:
# Residuals analysis
if use_lstm and hybrid_pred is not None:
    print("Analyzing residuals...")
    residuals = y_test_inv.flatten() - hybrid_pred
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Histogram
    ax1.hist(residuals, bins=50, color='#d35400', edgecolor='white', linewidth=1.2, alpha=0.7)
    ax1.axvline(x=residuals.mean(), color='red', linestyle='--', linewidth=2.5, label=f'Mean: {residuals.mean():.4f}')
    ax1.set_title("Residual Error Distribution", weight='bold', fontsize=13, pad=10)
    ax1.set_xlabel("Residuals (kWh)", weight='bold')
    ax1.set_ylabel("Frequency", weight='bold')
    ax1.legend(frameon=True, shadow=True)
    sns.despine(ax=ax1)
    
    # Q-Q plot
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=ax2)
    ax2.set_title("Q-Q Plot for Normality Check", weight='bold', fontsize=13, pad=10)
    ax2.grid(True, alpha=0.3)
    sns.despine(ax=ax2)
    
    print(f"Residual Statistics:")
    print(f"  Mean: {residuals.mean():.6f}")
    print(f"  Std Dev: {residuals.std():.6f}")
    print(f"  Min: {residuals.min():.6f}")
    print(f"  Max: {residuals.max():.6f}")
    
    plt.tight_layout()
    plt.savefig(save_plot("residual_analysis.png"), dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved residual analysis plot")
else:
    print("Skipping residuals analysis - hybrid model unavailable")

Analyzing residuals...
Residual Statistics:
  Mean: -0.002118
  Std Dev: 0.005001
  Min: -0.027482
  Max: 0.022810
Saved residual analysis plot


## Section 21: Anomaly Detection

# Anomaly Detection using Isolation Forest
print("Performing anomaly detection...")
iso = IsolationForest(contamination=0.01, random_state=42)
hourly_df_copy = hourly_df.copy()
hourly_df_copy["anomaly"] = iso.fit_predict(hourly_df_copy[["energy"]])

# Separate normal and anomalous points
normal_pts = hourly_df_copy[hourly_df_copy["anomaly"] == 1]
anomalous_pts = hourly_df_copy[hourly_df_copy["anomaly"] == -1]

print(f"Total points: {len(hourly_df_copy)}")
print(f"Normal points: {len(normal_pts)}")
print(f"Anomalies detected: {len(anomalous_pts)} ({100*len(anomalous_pts)/len(hourly_df_copy):.2f}%)")

# Plot
fig, ax = plt.subplots(figsize=(16, 6))
ax.scatter(normal_pts.index, normal_pts["energy"], color='#95a5a6', label='Normal', alpha=0.4, s=25, edgecolors='none')
ax.scatter(anomalous_pts.index, anomalous_pts["energy"], color='#c0392b', label='Anomaly', alpha=1.0, s=100, edgecolors='black', linewidth=0.8, marker='X')

ax.set_title("Energy Consumption Anomaly Detection (Isolation Forest)", weight='bold', fontsize=14, pad=15)
ax.set_xlabel("Date / Time Index", weight='bold', fontsize=12)
ax.set_ylabel("Energy Consumption (kWh)", weight='bold', fontsize=12)
ax.legend(frameon=True, shadow=True, fontsize=11, markerscale=1.2)
sns.despine()
plt.tight_layout()
plt.savefig(save_plot("anomaly_detection.png"), dpi=300, bbox_inches='tight')
plt.show()
print("✓Saved anomaly detection plot to output_plots/anomaly_detection.png")

# Print details of anomalies
if len(anomalous_pts) > 0:
    print("\nTop 10 Anomalies (by energy consumption):")
    print(anomalous_pts[["energy"]].nlargest(10, "energy"))

In [78]:
# Anomaly Detection using Isolation Forest
print("Performing anomaly detection...")
iso = IsolationForest(contamination=0.01, random_state=42)
hourly_df_copy = hourly_df.copy()
hourly_df_copy["anomaly"] = iso.fit_predict(hourly_df_copy[["energy"]])

# Separate normal and anomalous points
normal_pts = hourly_df_copy[hourly_df_copy["anomaly"] == 1]
anomalous_pts = hourly_df_copy[hourly_df_copy["anomaly"] == -1]

print(f"Total points: {len(hourly_df_copy)}")
print(f"Normal points: {len(normal_pts)}")
print(f"Anomalies detected: {len(anomalous_pts)} ({100*len(anomalous_pts)/len(hourly_df_copy):.2f}%)")

# Plot
fig, ax = plt.subplots(figsize=(16, 6))
ax.scatter(normal_pts.index, normal_pts["energy"], color='#95a5a6', label='Normal', alpha=0.4, s=25, edgecolors='none')
ax.scatter(anomalous_pts.index, anomalous_pts["energy"], color='#c0392b', label='Anomaly', alpha=1.0, s=100, edgecolors='black', linewidth=0.8, marker='X')

ax.set_title("Energy Consumption Anomaly Detection (Isolation Forest)", weight='bold', fontsize=14, pad=15)
ax.set_xlabel("Date / Time Index", weight='bold', fontsize=12)
ax.set_ylabel("Energy Consumption (kWh)", weight='bold', fontsize=12)
ax.legend(frameon=True, shadow=True, fontsize=11, markerscale=1.2)
sns.despine()
plt.tight_layout()
plt.savefig(save_plot("anomaly_detection.png"), dpi=300, bbox_inches='tight')
plt.show()
print("✓Saved anomaly detection plot to output_plots/anomaly_detection.png")

# Print details of anomalies
if len(anomalous_pts) > 0:
    print("\nTop 10 Anomalies (by energy consumption):")
    print(anomalous_pts[["energy"]].nlargest(10, "energy"))

Performing anomaly detection...
Total points: 19680
Normal points: 19484
Anomalies detected: 196 (1.00%)
✓Saved anomaly detection plot to output_plots/anomaly_detection.png

Top 10 Anomalies (by energy consumption):
                       energy
tstp                         
2012-02-05 19:00:00  0.482356
2013-01-20 18:00:00  0.470245
2012-02-12 19:00:00  0.466756
2012-02-02 19:00:00  0.465531
2013-01-20 19:00:00  0.463364
2012-02-05 18:00:00  0.459608
2012-01-31 19:00:00  0.459592
2013-01-18 19:00:00  0.457836
2012-02-10 19:00:00  0.457542
2012-01-30 19:00:00  0.456825


## Section 22: Comprehensive Results Summary

In [79]:
# Create comprehensive summary
print("\n" + "="*70)
print("COMPREHENSIVE ENERGY FORECASTING ANALYSIS SUMMARY")
print("="*70)

print(f"\nDATASET INFORMATION:")
print(f"  * Time Range: {hourly_df.index.min()} to {hourly_df.index.max()}")
print(f"  * Total Records: {len(hourly_df):,}")
print(f"  * Number of Features: {len(feature_list)}")
print(f"  * Energy Range: {hourly_df['energy'].min():.4f} - {hourly_df['energy'].max():.4f} kWh")
print(f"  * Mean Energy: {hourly_df['energy'].mean():.4f} kWh")
print(f"  * Std Dev Energy: {hourly_df['energy'].std():.4f} kWh")

print(f"\nTRADITIONAL ML MODELS PERFORMANCE:")
print(results_df.to_string())
print(f"\n  Best Model: {best_model_name}")
print(f"     - RMSE: {results_df.loc[best_model_name, 'RMSE']:.6f}")
print(f"     - MAE:  {results_df.loc[best_model_name, 'MAE']:.6f}")
print(f"     - R2:   {results_df.loc[best_model_name, 'R2']:.6f}")

if use_lstm and lstm_results is not None:
    print(f"\nLSTM MODEL PERFORMANCE:")
    print(f"  - RMSE: {lstm_results['RMSE']:.6f}")
    print(f"  - MAE:  {lstm_results['MAE']:.6f}")
    print(f"  - R2:   {lstm_results['R2']:.6f}")

if use_lstm and hybrid_results is not None:
    print(f"\nHYBRID MODEL PERFORMANCE ({best_model_name} + LSTM):")
    print(f"  - RMSE: {hybrid_results['RMSE']:.6f}")
    print(f"  - MAE:  {hybrid_results['MAE']:.6f}")
    print(f"  - R2:   {hybrid_results['R2']:.6f}")

print(f"\nOUTPUT FILES GENERATED:")
output_files = [
    "model_comparison_rmse.png",
    "model_comparison_mae.png",
    "model_comparison_combined.png",
    "feature_importance.png",
    "seasonal_decompose.png",
    "acf_plot.png",
    "load_duration_curve.png",
    "temp_vs_energy.png",
    "residual_analysis.png",
    "anomaly_detection.png",
    "hybrid_prediction.png" if use_lstm else None,
    "lstm_prediction.png" if use_lstm else None,
    "shap_summary.png"
]
for i, file in enumerate([f for f in output_files if f is not None], 1):
    print(f"  {i:2d}. {file}")

print(f"\n{'='*70}")


COMPREHENSIVE ENERGY FORECASTING ANALYSIS SUMMARY

DATASET INFORMATION:
  * Time Range: 2011-12-01 00:00:00 to 2014-02-27 23:00:00
  * Total Records: 19,680
  * Number of Features: 20
  * Energy Range: 0.0924 - 0.4824 kWh
  * Mean Energy: 0.2165 kWh
  * Std Dev Energy: 0.0760 kWh

TRADITIONAL ML MODELS PERFORMANCE:
                       MAE      RMSE        R2
LinearRegression  0.007348  0.009635  0.982938
RandomForest      0.005482  0.007555  0.989508
GradientBoosting  0.006463  0.008658  0.986220
XGBoost           0.004976  0.006943  0.991138

  Best Model: XGBoost
     - RMSE: 0.006943
     - MAE:  0.004976
     - R2:   0.991138

LSTM MODEL PERFORMANCE:
  - RMSE: 0.009841
  - MAE:  0.007606
  - R2:   0.983632

HYBRID MODEL PERFORMANCE (XGBoost + LSTM):
  - RMSE: 0.005431
  - MAE:  0.004205
  - R2:   0.995015

OUTPUT FILES GENERATED:
   1. model_comparison_rmse.png
   2. model_comparison_mae.png
   3. model_comparison_combined.png
   4. feature_importance.png
   5. seasonal_decompo

## Section 23: Export Results to CSV

In [80]:
# Export results to CSV for paper
results_df.to_csv(save_plot("model_performance_comparison.csv"))
print("Exported model performance comparison")

if hasattr(best_model, 'feature_importances_'):
    feat_df.to_csv(save_plot("feature_importance.csv"), index=False)
    print("Exported feature importance")

if use_lstm:
    # Create comparison of all models
    all_results = results_df.copy()
    all_results.loc['LSTM'] = lstm_results
    all_results.loc['Hybrid'] = hybrid_results
    all_results.to_csv(save_plot("all_models_performance.csv"))
    print("Exported all models performance")
    print(f"\nAll Models Summary:")
    print(all_results)

Exported model performance comparison
Exported feature importance
Exported all models performance

All Models Summary:
                       MAE      RMSE        R2
LinearRegression  0.007348  0.009635  0.982938
RandomForest      0.005482  0.007555  0.989508
GradientBoosting  0.006463  0.008658  0.986220
XGBoost           0.004976  0.006943  0.991138
LSTM              0.007606  0.009841  0.983632
Hybrid            0.004205  0.005431  0.995015
